# Chicago Community Area Boundaries

## Purpose

Download and validate Chicago's official community area boundaries.

This notebook will confirm that all 77 community areas are there and creates a reference table for later mapping. It doesn't
 create the final map.

In [7]:
from pathlib import Path
import json

import pandas as pd
import requests

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

assert (project_root / "notebooks").is_dir(), (
    f"Project root was not found: {project_root}"
)

raw_boundary_path = (
    project_root
    / "data"
    / "raw"
    / "chicago_community_areas.geojson"
)

reference_output_path = (
    project_root
    / "data"
    / "processed"
    / "community_area_reference.csv"
)

print("Project root:", project_root)

Project root: /workspaces/chicago-homelessness-resource-allocation/chicago-homelessness-resource-allocation


In [8]:
boundary_url = (
    "https://data.cityofchicago.org/resource/"
    "igwz-8jzy.geojson?$limit=5000"
)

response = requests.get(
    boundary_url,
    timeout=60,
)

response.raise_for_status()
boundary_geojson = response.json()

assert boundary_geojson.get("type") == "FeatureCollection", (
    "The response is not a GeoJSON FeatureCollection."
)

features = boundary_geojson.get("features", [])

print("HTTP status:", response.status_code)
print("Downloaded boundary features:", len(features))

HTTP status: 200
Downloaded boundary features: 77


In [9]:
assert len(features) == 77, (
    f"Expected 77 community areas but found {len(features)}."
)

assert all(
    feature.get("geometry") is not None
    for feature in features
), "At least one community area has no geometry."

geometry_types = {
    feature["geometry"]["type"]
    for feature in features
}

assert geometry_types.issubset(
    {"Polygon", "MultiPolygon"}
), f"Unexpected geometry types: {geometry_types}"

boundary_properties = pd.json_normalize(
    [feature["properties"] for feature in features]
)

required_properties = {
    "area_numbe",
    "community",
}

missing_properties = (
    required_properties - set(boundary_properties.columns)
)

assert not missing_properties, (
    f"Missing boundary properties: {missing_properties}"
)

print("Boundary validation passed.")
print("Geometry types:", sorted(geometry_types))

display(boundary_properties.head())

Boundary validation passed.
Geometry types: ['MultiPolygon']


,community,shape_area,area_num_1,area_numbe,shape_len
0,ROGERS PARK,51259902.4506,1,1,34052.3975757
1,WEST RIDGE,98429094.8621,2,2,43020.6894583
2,UPTOWN,65095642.7289,3,3,46972.7945549
3,LINCOLN SQUARE,71352328.2399,4,4,36624.6030848
4,NORTH CENTER,57054167.85,5,5,31391.6697542


In [10]:
boundary_reference = pd.DataFrame(
    {
        "community_area_number": pd.to_numeric(
            boundary_properties["area_numbe"],
            errors="coerce",
        ).astype("Int64"),
        "official_community_area": (
            boundary_properties["community"]
            .astype("string")
            .str.strip()
        ),
    }
).sort_values(
    "community_area_number"
).reset_index(drop=True)

assert boundary_reference[
    "community_area_number"
].notna().all()

assert boundary_reference[
    "community_area_number"
].is_unique

assert set(
    boundary_reference["community_area_number"]
) == set(range(1, 78)), (
    "Community-area numbers are not exactly 1 through 77."
)

assert boundary_reference[
    "official_community_area"
].notna().all()

print("Community area reference validation passed.")
display(boundary_reference)

Community area reference validation passed.


,community_area_number,official_community_area
0,1,ROGERS PARK
1,2,WEST RIDGE
2,3,UPTOWN
3,4,LINCOLN SQUARE
4,5,NORTH CENTER
...,...,...
72,73,WASHINGTON HEIGHTS
73,74,MOUNT GREENWOOD
74,75,MORGAN PARK
75,76,OHARE


In [11]:
housing_summary_path = (
    project_root
    / "data"
    / "processed"
    / "affordable_housing_by_community_area.csv"
)

housing_summary = pd.read_csv(housing_summary_path)

housing_summary["community_area_number"] = pd.to_numeric(
    housing_summary["community_area_number"],
    errors="coerce",
).astype("Int64")

assert housing_summary[
    "community_area_number"
].notna().all(), (
    "The housing summary contains a missing or invalid area number."
)

boundary_numbers = set(
    boundary_reference["community_area_number"]
)

housing_numbers = set(
    housing_summary["community_area_number"]
)

invalid_housing_numbers = housing_numbers - boundary_numbers

assert not invalid_housing_numbers, (
    "Housing summary contains invalid community-area numbers: "
    f"{sorted(invalid_housing_numbers)}"
)

areas_without_housing_records = (
    boundary_numbers - housing_numbers
)

print(
    "Areas represented in housing summary:",
    len(housing_numbers),
)
print(
    "Areas without documented housing records:",
    len(areas_without_housing_records),
)
print(
    "Area numbers without documented records:",
    sorted(areas_without_housing_records),
)

Areas represented in housing summary: 66
Areas without documented housing records: 11
Area numbers without documented records: [np.int64(12), np.int64(20), np.int64(47), np.int64(52), np.int64(57), np.int64(59), np.int64(64), np.int64(72), np.int64(74), np.int64(75), np.int64(76)]


In [12]:
flags_path = (
    project_root
    / "docs"
    / "GEOGRAPHY_REVIEW_FLAGS.csv"
)

geography_flags = pd.read_csv(flags_path)

number_columns = [
    "listed_community_area_number",
    "coordinate_community_area_number",
]

for column in number_columns:
    geography_flags[column] = pd.to_numeric(
        geography_flags[column],
        errors="coerce",
    ).astype("Int64")

assert len(geography_flags) == 7, (
    f"Expected 7 geography flags but found "
    f"{len(geography_flags)}."
)

assert geography_flags[number_columns].notna().all().all(), (
    "A flagged record contains a missing area number."
)

flag_numbers = set(
    geography_flags["listed_community_area_number"]
).union(
    geography_flags["coordinate_community_area_number"]
)

assert flag_numbers.issubset(boundary_numbers), (
    "A flagged record contains an invalid area number."
)

print("Geography flags validated:", len(geography_flags))
display(geography_flags)


Geography flags validated: 7


,property_name,address,listed_community_area,listed_community_area_number,coordinate_community_area,coordinate_community_area_number,reason,verification_source,reviewed_on
0,Homan Square Apartments Phase IV,908 S. Central Park Ave.,East Garfield Park,27,North Lawndale,29,"Retain listed area pending review of address, ...",https://data.cityofchicago.org/d/igwz-8jzy,2026-10-21
1,The Rebecca Johnson Apts.,2822 W. Jackson Blvd.,West Garfield Park,26,East Garfield Park,27,"Retain listed area pending review of address, ...",https://data.cityofchicago.org/d/igwz-8jzy,2026-10-21
2,Homan Square Apartments Phase VI,3609 W. Arthington St.,East Garfield Park,27,North Lawndale,29,"Retain listed area pending review of address, ...",https://data.cityofchicago.org/d/igwz-8jzy,2026-10-21
3,The Medallion,3121 N. Broadway,Lake View,6,Uptown,3,"Retain listed area pending review of address, ...",https://data.cityofchicago.org/d/igwz-8jzy,2026-10-21
4,Lawndale Restoration II Apts. (scattered sites),10 N. Hamlin Ave.,North Lawndale,29,West Garfield Park,26,"Retain listed area pending review of address, ...",https://data.cityofchicago.org/d/igwz-8jzy,2026-10-21
5,The Ludlow,2032 N. Clybourn Ave.,Lincoln Park,7,Near North Side,8,"Retain listed area pending review of address, ...",https://data.cityofchicago.org/d/igwz-8jzy,2026-10-21
6,Dorchester Art + Housing Collaborative,1450 E. 70th St.,Greater Grand Crossing,69,South Shore,43,"Retain listed area pending review of address, ...",https://data.cityofchicago.org/d/igwz-8jzy,2026-10-21


In [13]:
raw_boundary_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

reference_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

raw_boundary_path.write_text(
    json.dumps(boundary_geojson),
    encoding="utf-8",
)

boundary_reference.to_csv(
    reference_output_path,
    index=False,
)

saved_reference = pd.read_csv(
    reference_output_path
)

assert raw_boundary_path.is_file()
assert raw_boundary_path.stat().st_size > 0
assert len(saved_reference) == 77

pd.testing.assert_frame_equal(
    boundary_reference.reset_index(drop=True),
    saved_reference,
    check_dtype=False,
)

print("Community boundary preparation passed.")
print("Raw boundary file:", raw_boundary_path)
print("Reference file:", reference_output_path)

Community boundary preparation passed.
Raw boundary file: /workspaces/chicago-homelessness-resource-allocation/chicago-homelessness-resource-allocation/data/raw/chicago_community_areas.geojson
Reference file: /workspaces/chicago-homelessness-resource-allocation/chicago-homelessness-resource-allocation/data/processed/community_area_reference.csv
